# Proyecto LSP — Pipeline completo en Colab

Notebook end-to-end del proyecto **Reconocimiento de Lengua de Señas Peruana** (UPAO, Percepción Computacional 2026-1).

Este notebook orquesta el pipeline en Colab pero **toda la lógica vive en el repo** (módulos de `src/` + `config.py`). Si actualizas el código en GitHub, basta con re-clonar.

**Flujo:**
1. Clona el repo y configura el entorno.
2. Extrae los `.rar` del dataset VideoLSP10.
3. Procesa frames → keypoints con `src/preprocesamiento/extraer_dataset.py`.
4. Carga y divide el dataset con `src/utils/datos.py`.
5. Entrena baseline SVM con `src/modelos/baseline_svm.py`.
6. Entrena modelo LSTM con `src/entrenamiento/entrenar.py`.
7. Evalúa con `src/evaluacion/evaluar.py`.
8. Guarda `modelo_lsp_final.keras` en Drive.

**Pre-requisitos:**
- Activar **GPU T4** (`Entorno de ejecución` → `Cambiar tipo de hardware` → `T4 GPU`).
- Subir los 6 archivos `rgb.partN.rar` a `Drive/VideoLSP10_rar/`.

## Setup — clonar repo, instalar y montar Drive

In [ ]:
# Clonar el repositorio (idempotente: limpia si ya estaba)
!rm -rf /content/Proyecto_Percepcion
!git clone https://github.com/Jtarazona00/Proyecto_Percepcion.git /content/Proyecto_Percepcion
%cd /content/Proyecto_Percepcion

# Versiones compatibles entre si:
# - mediapipe 0.10.14: ultima con mp.solutions (requiere protobuf 4.x)
# - tensorflow 2.18: compatible con protobuf 4.x y Python 3.12 de Colab
!apt-get install -y unrar -qq
!pip install -q tensorflow==2.18.0 mediapipe==0.10.14

# Montar Drive y verificar GPU
import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')
print('GPU disponible:', tf.config.list_physical_devices('GPU'))
print('TF version:', tf.__version__)

## Paso 1 — Configuración

Las **clases, frames y features** se importan desde `config.py` del repo. Solo definimos aquí las **rutas específicas de Colab/Drive**.

In [ ]:
import sys
sys.path.insert(0, '/content/Proyecto_Percepcion')

from pathlib import Path
from config import CLASSES, NUM_CLASSES, FRAMES, FEATURES_PER_FRAME

# Rutas en Drive (ajusta si tienes otras)
RAR_DIR = Path('/content/drive/MyDrive/VideoLSP10_rar')
PROCESSED_DIR_DRIVE = Path('/content/drive/MyDrive/VideoLSP10_processed')
MODELS_DIR_DRIVE = Path('/content/drive/MyDrive/VideoLSP10_models')

# Rutas temporales en disco local de Colab (más rápidas)
LOCAL_RAR = Path('/content/rar_files')
LOCAL_EXTRACTED = Path('/content/rgb_extracted')

for d in (PROCESSED_DIR_DRIVE, MODELS_DIR_DRIVE, LOCAL_RAR, LOCAL_EXTRACTED):
    d.mkdir(parents=True, exist_ok=True)

rars = sorted(RAR_DIR.glob('rgb.part*.rar'))
print(f'{len(rars)} archivos .rar en {RAR_DIR}:')
for r in rars:
    print(f'  - {r.name} ({r.stat().st_size / 1024 / 1024:.0f} MB)')
assert len(rars) == 6, 'Deberías tener exactamente 6 archivos rgb.partN.rar en Drive'
print(f'\n{NUM_CLASSES} clases configuradas en config.py: {CLASSES}')

## Paso 2 — Extraer los `.rar` en disco local de Colab

Copia los `.rar` de Drive al disco temporal de Colab (más rápido) y los descomprime. Tarda 5-15 minutos.

In [ ]:
import shutil

for r in rars:
    destino = LOCAL_RAR / r.name
    if not destino.exists():
        shutil.copy(r, destino)
        print(f'  copiado: {r.name}')

print('\nExtrayendo (5-15 min)...')
!unrar x -o+ "{LOCAL_RAR}/rgb.part1.rar" "{LOCAL_EXTRACTED}/" > /tmp/unrar.log 2>&1 && echo '   OK' || (echo '   ERROR:'; tail -20 /tmp/unrar.log)

## Paso 3 — Procesar frames → keypoints con MediaPipe

Toda la lógica vive en `src/preprocesamiento/extraer_dataset.py`. El notebook solo invoca la función y le pasa rutas.

**Es resumible**: si Colab se desconecta, al re-ejecutar continúa donde quedó (los `.npy` se guardan directo en Drive).

In [ ]:
from src.preprocesamiento.extraer_dataset import parse_nombre_carpeta, procesar_dataset

# Detectar la carpeta donde estan las secuencias <clase>_r.<id>/
candidatos = [LOCAL_EXTRACTED] + [p for p in LOCAL_EXTRACTED.iterdir() if p.is_dir()]
BASE_RGB = None
for c in candidatos:
    for sub in c.iterdir():
        if sub.is_dir() and parse_nombre_carpeta(sub.name)[0] in CLASSES:
            BASE_RGB = c
            break
    if BASE_RGB:
        break

assert BASE_RGB is not None, 'No se encontraron carpetas <clase>_r.<id>/'
print(f'Carpeta base detectada: {BASE_RGB}')

# Procesar todo (resumible)
procesar_dataset(BASE_RGB, salida=PROCESSED_DIR_DRIVE)

## Paso 4 — Cargar dataset y split estratificado 80/10/10

Usa `src/utils/datos.py`.

In [ ]:
from src.utils.datos import cargar_dataset, particionar

X, y = cargar_dataset(processed_dir=PROCESSED_DIR_DRIVE)
print(f'Dataset cargado: X={X.shape}, y={y.shape}')

X_train, X_val, X_test, y_train, y_val, y_test = particionar(X, y)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## Paso 5 — Baseline SVM (Sección 4.1)

Usa `src/modelos/baseline_svm.py`: features estadísticas (media + std + rango → 774) + SVM RBF.

In [ ]:
from src.modelos.baseline_svm import features_dataset, entrenar_svm

X_train_stats = features_dataset(X_train)
X_test_stats  = features_dataset(X_test)

svm, scaler, precision_baseline = entrenar_svm(
    X_train_stats, y_train, X_test_stats, y_test
)

## Paso 6 — Entrenar modelo LSTM (Secciones 4.2 y 4.3)

Arquitectura definida en `src/modelos/lstm_modelo.py`. Entrenamiento (Adam 0.0005, 100 épocas, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau) en `src/entrenamiento/entrenar.py`.

In [ ]:
from src.entrenamiento.entrenar import entrenar

model, historial = entrenar(X_train, y_train, X_val, y_val)

## Paso 7 — Evaluación (Sección 5)

Usa `src/evaluacion/evaluar.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.evaluacion.evaluar import evaluar_modelo, matriz_confusion, graficar_historial

graficar_historial(historial)
plt.savefig('curvas_aprendizaje.png', dpi=120, bbox_inches='tight')

y_pred = evaluar_modelo(model, X_test, y_test)
matriz_confusion(y_test, y_pred, guardar='matriz_confusion.png')

precision_lstm = float((y_pred == y_test).mean())
print(f'\n=== Comparacion ===')
print(f'  Baseline SVM: {precision_baseline:.2%}')
print(f'  Modelo LSTM:  {precision_lstm:.2%}')

## Paso 8 — Guardar modelo y gráficas en Drive

In [ ]:
import shutil

modelo_final = MODELS_DIR_DRIVE / 'modelo_lsp_final.keras'
shutil.copy('modelo_lsp_final.keras', modelo_final)
shutil.copy('curvas_aprendizaje.png', MODELS_DIR_DRIVE / 'curvas_aprendizaje.png')
shutil.copy('matriz_confusion.png',  MODELS_DIR_DRIVE / 'matriz_confusion.png')

print(f'Modelo y graficas guardados en: {MODELS_DIR_DRIVE}')

from google.colab import files
files.download(str(modelo_final))